In [ ]:
# Import

import pandas as pd
import numpy as np
from scipy.optimize import minimize
from scipy.special import comb

# Konstansok
TOTAL_NUMBERS = 90
NUMBERS_DRAWN = 5

# Kombinációs valószínűségek
TOTAL_COMBINATIONS = comb(TOTAL_NUMBERS, NUMBERS_DRAWN, exact=True)
P_5 = 1 / TOTAL_COMBINATIONS
P_4 = comb(5, 4) * comb(85, 1) / TOTAL_COMBINATIONS
P_3 = comb(5, 3) * comb(85, 2) / TOTAL_COMBINATIONS
P_2 = comb(5, 2) * comb(85, 3) / TOTAL_COMBINATIONS

print(f"Valószínűségek:")
print(f"5 találat: 1/{TOTAL_COMBINATIONS:,} = {P_5:.10f}")
print(f"4 találat: {P_4:.6f}")
print(f"3 találat: {P_3:.6f}")
print(f"2 találat: {P_2:.6f}")

In [ ]:
# ADATOK BETÖLTÉSE ÉS TISZTÍTÁSA

huzasok = pd.read_csv("https://bet.szerencsejatek.hu/cmsfiles/otos.csv", 
                      sep=";", header=None, 
                      names=["year", "week", "date", "fives", "fives_prize", 
                             "fours", "fours_prize", "threes", "threes_prize", 
                             "twos", "twos_prize", "n1", "n2", "n3", "n4", "n5"])

# Szelvényárak
szelo_arak = {
    '1957.01.': 130,
    '2003.04.': 150,
    '2005.03.': 175,
    '2007.11.': 200,
    '2010.02.': 225,
    '2016.11.': 250,
    '2020.01.': 300,
    '2022.10.': 350,
    '2023.09.': 400
}

def get_ticket_price(date_str):
    """Meghatározza a szelvény árát az adott dátumra"""
    if pd.isna(date_str):
        return 130
    
    for date_threshold, price in sorted(szelo_arak.items(), reverse=True):
        year = int(date_threshold.split('.')[0])
        month = int(date_threshold.split('.')[1])
        
        try:
            draw_year = int(date_str.split('.')[0])
            draw_month = int(date_str.split('.')[1])
            
            if draw_year > year or (draw_year == year and draw_month >= month):
                return price
        except:
            return 130
    
    return 130

# Szelvényár hozzáadása
huzasok['ticket_price'] = huzasok['date'].apply(get_ticket_price)

# Nyeremények tisztítása (eltávolítjuk az "Ft" és szóközöket, átalakítjuk számmá)
def clean_prize(prize_str):
    if pd.isna(prize_str) or prize_str == '0 Ft' or prize_str == 0:
        return 0
    try:
        return int(str(prize_str).replace('Ft', '').replace(' ', '').replace(',', ''))
    except:
        return 0

for col in ['fives_prize', 'fours_prize', 'threes_prize', 'twos_prize']:
    huzasok[col] = huzasok[col].apply(clean_prize)

# Nyeremények relativizálása (hányszorosa a szelvényárnak)
huzasok['rel_fives_prize'] = huzasok['fives_prize'] / huzasok['ticket_price']
huzasok['rel_fours_prize'] = huzasok['fours_prize'] / huzasok['ticket_price']
huzasok['rel_threes_prize'] = huzasok['threes_prize'] / huzasok['ticket_price']
huzasok['rel_twos_prize'] = huzasok['twos_prize'] / huzasok['ticket_price']

# Csak azokat hagyjuk, ahol van adat
huzasok_clean = huzasok[
    (huzasok['twos'] > 0) & 
    (huzasok['threes'] > 0) & 
    (huzasok['fours'] > 0)
].copy()

print(f"\n\nÖsszesen {len(huzasok)} sorsolás, ebből felhasználható: {len(huzasok_clean)}")

In [ ]:
# JÁTÉKOSSZÁM BECSLÉSE

print("\n" + "="*80)
print("JÁTÉKOSSZÁM BECSLÉSE")
print("="*80)

def estimate_players_from_category(winners, probability):
    """Játékosszám becslése egy kategóriából"""
    if winners == 0:
        return np.nan
    return winners / probability

# Becslés minden kategóriából
huzasok_clean['players_from_2'] = huzasok_clean['twos'].apply(
    lambda x: estimate_players_from_category(x, P_2))
huzasok_clean['players_from_3'] = huzasok_clean['threes'].apply(
    lambda x: estimate_players_from_category(x, P_3))
huzasok_clean['players_from_4'] = huzasok_clean['fours'].apply(
    lambda x: estimate_players_from_category(x, P_4))

# Súlyozott átlag (2-találatost preferáljuk, mert nagy mintaszám és stabil)
huzasok_clean['estimated_players'] = (
    huzasok_clean['players_from_2'] * 0.7 +
    huzasok_clean['players_from_3'] * 0.2 +
    huzasok_clean['players_from_4'] * 0.1
)

# Outlierek szűrése (pl. IQR módszer)
Q1 = huzasok_clean['estimated_players'].quantile(0.25)
Q3 = huzasok_clean['estimated_players'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

huzasok_clean['estimated_players'] = huzasok_clean['estimated_players'].clip(
    lower=lower_bound, upper=upper_bound)

print(f"\nBecsült játékosszám statisztikák:")
print(huzasok_clean['estimated_players'].describe())

# Időbeli trend
recent_data = huzasok_clean[huzasok_clean['year'] >= 2020]
print(f"\n2020 óta átlagos játékosszám: {recent_data['estimated_players'].mean():,.0f}")

In [ ]:
# SZÁMNÉPSZERŰSÉG BECSLÉSE

# ============================================================================
# SZÁMNÉPSZERŰSÉG BECSLÉSE - JAVÍTOTT VERZIÓ
# ============================================================================

print("\n" + "="*80)
print("SZÁMNÉPSZERŰSÉG BECSLÉSE (2, 3, 4 TALÁLATOSOK ALAPJÁN)")
print("="*80)

def calculate_number_popularity_from_category(huzasok_df, category='twos'):
    """
    Számnépszerűség becslése egy találati kategória alapján
    
    category: 'twos', 'threes', vagy 'fours'
    """
    
    # Kategória adatai
    if category == 'twos':
        winners_col = 'twos'
        probability = P_2
        match_count = 2
    elif category == 'threes':
        winners_col = 'threes'
        probability = P_3
        match_count = 3
    elif category == 'fours':
        winners_col = 'fours'
        probability = P_4
        match_count = 4
    else:
        raise ValueError("Invalid category")
    
    # Csak azokat a sorsolásokat, ahol van nyertes
    huzasok_cat = huzasok_df[huzasok_df[winners_col] > 0].copy()
    
    print(f"\n{category.upper()}: {len(huzasok_cat)} sorsolás elemezve")
    
    # Várható nyertesek száma, ha minden kombináció egyformán népszerű lenne
    huzasok_cat['expected_winners'] = (
        huzasok_cat['estimated_players'] * probability
    )
    
    # Megfigyelt / várható arány
    huzasok_cat['popularity_factor'] = (
        huzasok_cat[winners_col] / huzasok_cat['expected_winners']
    )
    
    # Outlier szűrés (túl extrém értékek)
    Q1 = huzasok_cat['popularity_factor'].quantile(0.1)
    Q3 = huzasok_cat['popularity_factor'].quantile(0.9)
    huzasok_cat = huzasok_cat[
        (huzasok_cat['popularity_factor'] >= Q1) & 
        (huzasok_cat['popularity_factor'] <= Q3)
    ]
    
    print(f"  Outlier szűrés után: {len(huzasok_cat)} sorsolás")
    print(f"  Átlagos popularity_factor: {huzasok_cat['popularity_factor'].mean():.3f}")
    
    # Számonkénti elemzés
    number_scores = {}
    
    for num in range(1, 91):
        # Sorsolások, ahol ez a szám benne volt a kihúzott 5-ben
        with_num = huzasok_cat[
            (huzasok_cat['n1'] == num) |
            (huzasok_cat['n2'] == num) |
            (huzasok_cat['n3'] == num) |
            (huzasok_cat['n4'] == num) |
            (huzasok_cat['n5'] == num)
        ]
        
        if len(with_num) >= 10:  # Minimum 10 előfordulás kell
            # Átlagos popularity factor, amikor ez a szám benne van
            avg_with = with_num['popularity_factor'].mean()
            # Átlagos popularity factor általában
            avg_overall = huzasok_cat['popularity_factor'].mean()
            
            # Relatív népszerűség
            number_scores[num] = avg_with / avg_overall
        else:
            number_scores[num] = None
    
    return number_scores, len(huzasok_cat)

# ============================================================================
# NÉPSZERŰSÉG BECSLÉSE MINDEN KATEGÓRIÁBÓL
# ============================================================================

popularity_twos, count_twos = calculate_number_popularity_from_category(
    huzasok_clean, 'twos')
popularity_threes, count_threes = calculate_number_popularity_from_category(
    huzasok_clean, 'threes')
popularity_fours, count_fours = calculate_number_popularity_from_category(
    huzasok_clean, 'fours')

# ============================================================================
# KOMBINÁLT NÉPSZERŰSÉG (SÚLYOZOTT ÁTLAG)
# ============================================================================

print("\n" + "="*80)
print("KOMBINÁLT NÉPSZERŰSÉG SZÁMÍTÁSA")
print("="*80)

# Súlyok: 2-találat a legtöbb adat, 4-találat a legkevesebb de legpontosabb
weights = {
    'twos': 0.5,      # Legnagyobb mintaszám
    'threes': 0.3,    # Közepes mintaszám
    'fours': 0.2      # Kis mintaszám, de direkt kapcsolat
}

number_popularity = {}

for num in range(1, 91):
    scores = []
    score_weights = []
    
    if popularity_twos[num] is not None:
        scores.append(popularity_twos[num])
        score_weights.append(weights['twos'])
    
    if popularity_threes[num] is not None:
        scores.append(popularity_threes[num])
        score_weights.append(weights['threes'])
    
    if popularity_fours[num] is not None:
        scores.append(popularity_fours[num])
        score_weights.append(weights['fours'])
    
    if scores:
        # Súlyozott átlag
        number_popularity[num] = np.average(scores, weights=score_weights)
    else:
        # Ha nincs elég adat, semleges 1.0
        number_popularity[num] = 1.0

# Normalizálás (átlag pontosan 1.0 legyen)
mean_pop = np.mean(list(number_popularity.values()))
number_popularity = {k: v/mean_pop for k, v in number_popularity.items()}

print(f"\nSzámonkénti népszerűség kiszámítva {len(number_popularity)} számra")
print(f"Átlag népszerűség (normalizálás után): {np.mean(list(number_popularity.values())):.6f}")
print(f"Szórás: {np.std(list(number_popularity.values())):.6f}")

# ============================================================================
# EREDMÉNYEK MEGJELENÍTÉSE
# ============================================================================

sorted_by_pop = sorted(number_popularity.items(), key=lambda x: x[1], reverse=True)

print("\n" + "="*80)
print("Top 15 LEGnépszerűbb szám:")
print("="*80)
for i, (num, pop) in enumerate(sorted_by_pop[:15], 1):
    # Részletes info
    info_parts = []
    if popularity_twos[num]:
        info_parts.append(f"2:{popularity_twos[num]:.3f}")
    if popularity_threes[num]:
        info_parts.append(f"3:{popularity_threes[num]:.3f}")
    if popularity_fours[num]:
        info_parts.append(f"4:{popularity_fours[num]:.3f}")
    
    info_str = ", ".join(info_parts)
    print(f"{i:2d}. Szám {num:2d}: {pop:.4f}× ({info_str})")

print("\n" + "="*80)
print("Top 15 LEGkevésbé népszerű szám:")
print("="*80)
for i, (num, pop) in enumerate(sorted_by_pop[-15:], 1):
    # Részletes info
    info_parts = []
    if popularity_twos[num]:
        info_parts.append(f"2:{popularity_twos[num]:.3f}")
    if popularity_threes[num]:
        info_parts.append(f"3:{popularity_threes[num]:.3f}")
    if popularity_fours[num]:
        info_parts.append(f"4:{popularity_fours[num]:.3f}")
    
    info_str = ", ".join(info_parts)
    print(f"{i:2d}. Szám {num:2d}: {pop:.4f}× ({info_str})")

# ============================================================================
# MINTÁZATOK ELEMZÉSE
# ============================================================================

print("\n" + "="*80)
print("MINTÁZATOK ELEMZÉSE")
print("="*80)

# Különböző tartományok
ranges = {
    '1-10': range(1, 11),
    '11-20': range(11, 21),
    '21-31 (születésnap vége)': range(21, 32),
    '32-40': range(32, 41),
    '41-50': range(41, 51),
    '51-60': range(51, 61),
    '61-70': range(61, 71),
    '71-80': range(71, 81),
    '81-90': range(81, 91),
}

print("\nNépszerűség tartományonként:")
for range_name, num_range in ranges.items():
    avg_pop = np.mean([number_popularity[n] for n in num_range])
    print(f"  {range_name:25s}: {avg_pop:.4f}×")

# Speciális csoportok
special_groups = {
    'Születésnapok (1-31)': range(1, 32),
    'Magas számok (60-90)': range(60, 91),
    'Kerek számok (×10)': [10, 20, 30, 40, 50, 60, 70, 80, 90],
    'Prímek (első 15)': [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47],
    'Páros számok': range(2, 91, 2),
    'Páratlan számok': range(1, 91, 2),
}

print("\nSpeciális csoportok:")
for group_name, numbers in special_groups.items():
    avg_pop = np.mean([number_popularity[n] for n in numbers])
    print(f"  {group_name:25s}: {avg_pop:.4f}×")

# Vizuális megjelenítés
print("\n" + "="*80)
print("HŐTÉRKÉP (számonkénti népszerűség)")
print("="*80)
print("\n1-45:")
for i in range(0, 45, 9):
    row = []
    for num in range(i+1, min(i+10, 46)):
        pop = number_popularity[num]
        if pop > 1.05:
            symbol = "██"  # Nagyon népszerű
        elif pop > 1.02:
            symbol = "▓▓"  # Népszerű
        elif pop > 0.98:
            symbol = "▒▒"  # Átlagos
        elif pop > 0.95:
            symbol = "░░"  # Kevésbé népszerű
        else:
            symbol = "  "  # Ritka
        row.append(f"{num:2d}{symbol}")
    print("  ".join(row))

print("\n46-90:")
for i in range(45, 90, 9):
    row = []
    for num in range(i+1, min(i+10, 91)):
        pop = number_popularity[num]
        if pop > 1.05:
            symbol = "██"
        elif pop > 1.02:
            symbol = "▓▓"
        elif pop > 0.98:
            symbol = "▒▒"
        elif pop > 0.95:
            symbol = "░░"
        else:
            symbol = "  "
        row.append(f"{num:2d}{symbol}")
    print("  ".join(row))

print("\nJelmagyarázat: ██ = nagyon népszerű, ▓▓ = népszerű, ▒▒ = átlagos, ░░ = kevésbé népszerű,    = ritka")

# ============================================================================
# KONZISZTENCIA ELLENŐRZÉS
# ============================================================================

print("\n" + "="*80)
print("KONZISZTENCIA ELLENŐRZÉS")
print("="*80)

# Mennyire korrelálnak a különböző kategóriák eredményei?
from scipy.stats import spearmanr

# Csak azok a számok, ahol mindhárom kategóriában van adat
common_numbers = [
    num for num in range(1, 91) 
    if all([
        popularity_twos[num] is not None,
        popularity_threes[num] is not None,
        popularity_fours[num] is not None
    ])
]

if len(common_numbers) > 20:
    scores_2 = [popularity_twos[n] for n in common_numbers]
    scores_3 = [popularity_threes[n] for n in common_numbers]
    scores_4 = [popularity_fours[n] for n in common_numbers]
    
    corr_23, p_23 = spearmanr(scores_2, scores_3)
    corr_24, p_24 = spearmanr(scores_2, scores_4)
    corr_34, p_34 = spearmanr(scores_3, scores_4)
    
    print(f"\nSpearman korreláció ({len(common_numbers)} közös szám):")
    print(f"  2-találat vs 3-találat: {corr_23:.3f} (p={p_23:.4f})")
    print(f"  2-találat vs 4-találat: {corr_24:.3f} (p={p_24:.4f})")
    print(f"  3-találat vs 4-találat: {corr_34:.3f} (p={p_34:.4f})")
    
    if all([corr_23 > 0.3, corr_24 > 0.3, corr_34 > 0.3]):
        print("  ✓ Konzisztens becslés - a különböző kategóriák hasonló mintázatot mutatnak")
    else:
        print("  ⚠ Alacsony korreláció - a becslés bizonytalanabb")

print("\n" + "="*80)
print("NÉPSZERŰSÉG BECSLÉS KÉSZ")
print("="*80)

In [ ]:
# VÁRHATÓ ÉRTÉK KALKULÁCIÓ

print("\n" + "="*80)
print("VÁRHATÓ ÉRTÉK KALKULÁCIÓ")
print("="*80)

# Átlagos nyeremények (relativizált) - csak pozitív értékek
avg_prize_4 = huzasok_clean[huzasok_clean['rel_fours_prize'] > 0]['rel_fours_prize'].mean()
avg_prize_3 = huzasok_clean[huzasok_clean['rel_threes_prize'] > 0]['rel_threes_prize'].mean()
avg_prize_2 = huzasok_clean[huzasok_clean['rel_twos_prize'] > 0]['rel_twos_prize'].mean()

print(f"\nÁtlagos nyeremények (szelvényár többszöröse):")
print(f"  4 találat: {avg_prize_4:.1f}× (kb {avg_prize_4 * 400:,.0f} Ft mai áron)")
print(f"  3 találat: {avg_prize_3:.1f}× (kb {avg_prize_3 * 400:,.0f} Ft mai áron)")
print(f"  2 találat: {avg_prize_2:.1f}× (kb {avg_prize_2 * 400:,.0f} Ft mai áron)")

# 5-találat becslése
# Problémák a huzasok_with_5 használatával:
# 1. Csak azokat látjuk, ahol VOLT nyertes (bias)
# 2. Nem látjuk a felhalmozódott jackpotokat
# 3. Kis mintaszám

# JOBB MEGKÖZELÍTÉS: Becsüljük a teljes nyereményalapot
# A Szerencsejáték Zrt. kb. 50%-ot fizet vissza nyereményekben

print("\n" + "-"*80)
print("5-TALÁLAT JACKPOT BECSLÉSE")
print("-"*80)

# Számítsuk ki a teljes befizetést és nyereménykiadást
huzasok_recent = huzasok_clean[huzasok_clean['year'] >= 2020].copy()
print(f"\nElemzés 2020 óta: {len(huzasok_recent)} sorsolás")

# Átlagos játékosszám
avg_players_recent = huzasok_recent['estimated_players'].mean()
print(f"Átlagos játékosszám: {avg_players_recent:,.0f}")

# Teljes befizetés egy sorsoláson (relativizált, szelvényár többszöröse)
total_revenue_per_draw = avg_players_recent  # minden szelvény = 1× a szelvényár

# Kifizetési arány becslése a megfigyelt nyereményekből
# Számítsuk ki, mennyi ment 2, 3, 4 találatosokra
def calculate_payout_for_category(df, winners_col, prize_col, probability):
    """Átlagos kifizetés egy kategóriára (szelvényár többszöröse)"""
    total_payout = (df[winners_col] * df[prize_col]).sum()
    total_tickets = df['estimated_players'].sum()
    payout_per_ticket = total_payout / total_tickets
    return payout_per_ticket

payout_2 = calculate_payout_for_category(huzasok_recent, 'twos', 'rel_twos_prize', P_2)
payout_3 = calculate_payout_for_category(huzasok_recent, 'threes', 'rel_threes_prize', P_3)
payout_4 = calculate_payout_for_category(huzasok_recent, 'fours', 'rel_fours_prize', P_4)

print(f"\nÁtlagos kifizetés szelvényenként:")
print(f"  2 találat: {payout_2:.6f}× ({payout_2*100:.4f}%)")
print(f"  3 találat: {payout_3:.6f}× ({payout_3*100:.4f}%)")
print(f"  4 találat: {payout_4:.6f}× ({payout_4*100:.4f}%)")
print(f"  2+3+4 összesen: {(payout_2+payout_3+payout_4):.6f}× ({(payout_2+payout_3+payout_4)*100:.2f}%)")

# Ha feltételezzük, hogy ~50% a kifizetési arány, akkor a maradék 5-találatra megy
assumed_total_payout_ratio = 0.50
remaining_for_jackpot = assumed_total_payout_ratio - (payout_2 + payout_3 + payout_4)

print(f"\nHa a teljes kifizetési arány {assumed_total_payout_ratio*100:.0f}%:")
print(f"  5-találatra marad: {remaining_for_jackpot:.6f}× ({remaining_for_jackpot*100:.2f}%)")

# Ez az "átlagos" jackpot egy sorsoláson
estimated_avg_jackpot_per_draw = remaining_for_jackpot * avg_players_recent

print(f"\n→ Becsült átlagos teljes jackpot/sorsolás: {estimated_avg_jackpot_per_draw:.1f}×")
print(f"  (kb {estimated_avg_jackpot_per_draw * 400:,.0f} Ft mai áron)")

# Validálás: nézzük meg azokat a sorsolásokat, ahol volt 5-találatos
huzasok_with_5_recent = huzasok_recent[huzasok_recent['fives'] > 0].copy()
if len(huzasok_with_5_recent) > 0:
    # Teljes jackpot = nyertesek × nyeremény/fő
    huzasok_with_5_recent['total_jackpot'] = (
        huzasok_with_5_recent['fives'] * huzasok_with_5_recent['rel_fives_prize']
    )
    observed_avg_jackpot = huzasok_with_5_recent['total_jackpot'].mean()
    avg_winners = huzasok_with_5_recent['fives'].mean()
    
    print(f"\nValidálás ({len(huzasok_with_5_recent)} sorsolás volt 5-találatossal):")
    print(f"  Megfigyelt átlag jackpot: {observed_avg_jackpot:.1f}×")
    print(f"  Átlagos nyertesszám: {avg_winners:.2f}")
    print(f"  Különbség: {abs(observed_avg_jackpot - estimated_avg_jackpot_per_draw):.1f}× "
          f"({abs(observed_avg_jackpot - estimated_avg_jackpot_per_draw)/estimated_avg_jackpot_per_draw*100:.1f}%)")
    
    # Használjuk a megfigyelt értéket, ha van elég adat
    if len(huzasok_with_5_recent) > 20:
        print(f"  → Megfigyelt érték használata (elég nagy minta)")
        estimated_total_jackpot = observed_avg_jackpot
    else:
        print(f"  → Becsült érték használata (kis minta: {len(huzasok_with_5_recent)})")
        estimated_total_jackpot = estimated_avg_jackpot_per_draw
else:
    print(f"\nNincs 5-találatos a legutóbbi adatokban, becslést használjuk")
    estimated_total_jackpot = estimated_avg_jackpot_per_draw

print(f"\n{'='*80}")
print(f"VÉGSŐ BECSLÉS - Átlagos teljes jackpot: {estimated_total_jackpot:.1f}×")
print(f"                (kb {estimated_total_jackpot * 400:,.0f} Ft mai áron)")
print(f"{'='*80}")

# ============================================================================
# VÁRHATÓ ÉRTÉK FÜGGVÉNY
# ============================================================================

def calculate_ev(numbers, number_popularity, avg_players, 
                 jackpot_multiplier=1.0, verbose=False):
    """
    Várható érték számítása egy adott számkombinációra
    
    Parameters:
    -----------
    numbers : list
        5 szám listája (1-90)
    number_popularity : dict
        Számonkénti népszerűség (1.0 = átlag)
    avg_players : float
        Átlagos játékosszám egy sorsoláson
    jackpot_multiplier : float
        Jackpot szorzó (ha felhalmozódott)
    verbose : bool
        Részletes kiírás
    
    Returns:
    --------
    dict: Várható érték komponensek
    """
    
    # Népszerűségi faktor ennek a kombinációnak
    # Feltétel: a számok népszerűsége független (szorzat modell)
    combo_popularity = np.prod([number_popularity[n] for n in numbers])
    
    # Becsült játékosszám ezzel a PONTOS kombinációval
    # P(valaki ezt a kombinációt játssza) = 1/C(90,5) × népszerűség
    base_rate = avg_players / TOTAL_COMBINATIONS
    expected_players_with_combo = base_rate * combo_popularity
    
    # Ha én is játszom, akkor összesen hányan játsszuk
    # (statisztikailag én is "része" vagyok a combo_popularity-nak, de gyakorlatilag +1)
    total_players_with_combo = expected_players_with_combo + 1
    
    # FONTOS JAVÍTÁS: Az expected_other_winners azt jelenti, hogy
    # "rajtam kívül hányan nyernének", de valójában a jackpot oszlik közöttünk
    # Ha 5-öt találok, akkor BIZTOSAN nyertem, a kérdés: hányan osztják meg velem?
    expected_co_winners = expected_players_with_combo  # rajtam kívül
    
    # Jackpot felosztva
    jackpot_total = estimated_total_jackpot * jackpot_multiplier
    jackpot_per_winner = jackpot_total / (expected_co_winners + 1)
    
    # Várható érték komponensek (szelvényár többszöröse)
    ev_5 = P_5 * jackpot_per_winner
    ev_4 = P_4 * avg_prize_4
    ev_3 = P_3 * avg_prize_3
    ev_2 = P_2 * avg_prize_2
    
    total_ev = ev_5 + ev_4 + ev_3 + ev_2 - 1.0  # -1 = szelvényár
    
    result = {
        'ev': total_ev,
        'ev_5': ev_5,
        'ev_4': ev_4,
        'ev_3': ev_3,
        'ev_2': ev_2,
        'combo_popularity': combo_popularity,
        'expected_co_winners': expected_co_winners,
        'expected_players_with_combo': expected_players_with_combo,
        'jackpot_per_winner': jackpot_per_winner,
        'jackpot_total': jackpot_total,
    }
    
    if verbose:
        print(f"\nKombináció: {numbers}")
        print(f"  Népszerűség: {combo_popularity:.4f}×")
        print(f"  Várható játékosok ezzel a kombóval: {expected_players_with_combo:.6f}")
        print(f"  Ha nyerek, társnyertesek: {expected_co_winners:.6f}")
        print(f"  Jackpot összesen: {jackpot_total:.1f}×")
        print(f"  Jackpot/fő: {jackpot_per_winner:.1f}×")
        print(f"  EV_5: {ev_5:.6f}×, EV_4: {ev_4:.6f}×, EV_3: {ev_3:.6f}×, EV_2: {ev_2:.6f}×")
        print(f"  → TELJES EV: {total_ev:.6f}× ({total_ev*400:.2f} Ft)")
    
    return result

# ============================================================================
# STRATÉGIÁK ÖSSZEHASONLÍTÁSA
# ============================================================================

print("\n" + "="*80)
print("VÁRHATÓ ÉRTÉK ÖSSZEHASONLÍTÁS")
print("="*80)

strategies = {
    'Legritkább számok': [87, 80, 88, 90, 75],  # Top 5 legkevésbé népszerű
    'Születésnapok': [7, 13, 19, 23, 31],       # Népszerű tartomány
    'Magas számok': [82, 85, 86, 87, 90],       # 81-90 tartomány
    'Vegyes optimalizált': [40, 62, 75, 87, 90], # Mix a legritkábbakból
    'Véletlen': [12, 28, 45, 63, 79],           # Kontroll
    'Legnépszerűbb': [5, 19, 73, 4, 9],         # Top 5 legnépszerűbb
}

avg_players = huzasok_clean['estimated_players'].mean()

print(f"\nÁtlagos játékosszám: {avg_players:,.0f} szelvény")
print(f"Becsült jackpot: {estimated_total_jackpot:.1f}× ({estimated_total_jackpot*400:,.0f} Ft)\n")

results = []
for name, numbers in strategies.items():
    result = calculate_ev(numbers, number_popularity, avg_players)
    results.append((name, numbers, result))
    
    print(f"{name}")
    print(f"  Számok: {numbers}")
    print(f"  Népszerűség: {result['combo_popularity']:.4f}×")
    print(f"  Várható társnyertesek (ha nyerek): {result['expected_co_winners']:.6f}")
    print(f"  Jackpot/fő: {result['jackpot_per_winner']:.1f}× ({result['jackpot_per_winner']*400:,.0f} Ft)")
    print(f"  EV komponensek:")
    print(f"    • 5 találat: {result['ev_5']:.6f}× ({result['ev_5']*400:.2f} Ft)")
    print(f"    • 4 találat: {result['ev_4']:.6f}× ({result['ev_4']*400:.2f} Ft)")
    print(f"    • 3 találat: {result['ev_3']:.6f}× ({result['ev_3']*400:.2f} Ft)")
    print(f"    • 2 találat: {result['ev_2']:.6f}× ({result['ev_2']*400:.2f} Ft)")
    print(f"  → TELJES EV: {result['ev']:.6f}× = {result['ev']*400:.2f} Ft/szelvény")
    print()

# Sorbarendezés EV szerint
results_sorted = sorted(results, key=lambda x: x[2]['ev'], reverse=True)

print("="*80)
print("RANGSOR (EV szerint):")
print("="*80)
for i, (name, numbers, result) in enumerate(results_sorted, 1):
    improvement_vs_worst = (result['ev'] - results_sorted[-1][2]['ev']) * 400
    print(f"{i}. {name:25s} | EV: {result['ev']*400:+7.2f} Ft | "
          f"Javulás: {improvement_vs_worst:+6.2f} Ft")

best = results_sorted[0]
worst = results_sorted[-1]
improvement = (best[2]['ev'] - worst[2]['ev']) * 400

print(f"\n→ LEGJOBB vs LEGROSSZABB különbség: {improvement:.2f} Ft/szelvény")
print(f"   ({improvement/400*100:.2f}% javulás a várható veszteségben)")


In [ ]:
P_2*3500 + P_3*37_000 + P_4*3_600_000 + P_5*3_200_000_000

In [ ]:
# JÁTÉKOSSZÁM BECSLÉSE JACKPOT FELHALMOZÓDÁSBÓL
# ============================================================================

print("\n" + "="*80)
print("JÁTÉKOSSZÁM BECSLÉSE JACKPOT FELHALMOZÓDÁSBÓL")
print("="*80)

# ============================================================================
# 1. JACKPOT FELHALMOZÓDÁSI SOROZATOK AZONOSÍTÁSA
# ============================================================================

# Keressük meg azokat a periódusokat, ahol több héten át nem volt 5-találatos
huzasok_sorted = huzasok.sort_values(['year', 'week']).reset_index(drop=True)
huzasok_sorted = huzasok_sorted[pd.notna(huzasok_sorted['week'])]

# Jackpot felhalmozódási sorozatok
accumulation_periods = []
current_period = []

for idx, row in huzasok_sorted.iterrows():
    if row['fives'] == 0:
        # Nincs nyertes, felhalmozódás folytatódik
        current_period.append(idx)
    else:
        # Van nyertes, lezárul a periódus
        if len(current_period) > 0:
            # Hozzáadjuk a nyertes hetet is (ez fizeti ki a felhalmozott jackpotot)
            current_period.append(idx)
            accumulation_periods.append(current_period)
            current_period = []

# Ha a végén van befejezetlen periódus
if len(current_period) > 0:
    accumulation_periods.append(current_period)

print(f"\nTalált {len(accumulation_periods)} jackpot felhalmozódási periódus")

# Csak azokat tartjuk meg, ahol legalább 2 hét volt (1 hét felhalmozódás + 1 hét kifizetés)
valid_periods = [p for p in accumulation_periods if len(p) >= 2]
print(f"Ebből {len(valid_periods)} legalább 2 hetes (használható)")

# ============================================================================
# 2. HETI BEVÉTEL BECSLÉSE FELHALMOZÓDÁSBÓL
# ============================================================================

def estimate_weekly_revenue_from_jackpot(period_indices, huzasok_df):
    """
    Becsüljük a heti bevételt egy jackpot felhalmozódási periódusból
    
    Logika:
    - Ha N hétig nincs nyertes, a jackpot N × (heti bevétel × jackpot_ratio)
    - A végső héten kifizetett jackpot = ez a felhalmozott összeg
    """
    
    if len(period_indices) < 2:
        return None
    
    # Utolsó hét (amikor kifizették)
    final_idx = period_indices[-1]
    final_row = huzasok_df.loc[final_idx]
    
    if final_row['fives'] == 0 or final_row['fives_prize'] == 0:
        return None
    
    # Teljes kifizetett jackpot
    total_jackpot_paid = final_row['fives'] * final_row['fives_prize']
    
    # Hány hét felhalmozódás volt (+ a kifizetés hete)
    weeks_accumulated = len(period_indices)
    
    # Szelvényár az adott időszakban
    ticket_price = final_row['ticket_price']
    
    # Jackpot relatív (szelvényár többszöröse)
    jackpot_relative = total_jackpot_paid / ticket_price
    
    # Becsült heti bevétel (szelvények száma)
    # Feltételezés: a bevétel X%-a megy jackpotra (ezt kell becsülni)
    # jackpot_relative = weeks × players × jackpot_ratio
    # players = jackpot_relative / (weeks × jackpot_ratio)
    
    return {
        'period_indices': period_indices,
        'weeks': weeks_accumulated,
        'year': final_row['year'],
        'final_week': final_row['week'],
        'jackpot_paid_ft': total_jackpot_paid,
        'jackpot_relative': jackpot_relative,
        'ticket_price': ticket_price,
        'winners': final_row['fives'],
    }

# Gyűjtsük össze az összes periódust
jackpot_data = []
for period in valid_periods:
    result = estimate_weekly_revenue_from_jackpot(period, huzasok_sorted)
    if result is not None:
        jackpot_data.append(result)

jackpot_df = pd.DataFrame(jackpot_data)
print(f"\n{len(jackpot_df)} használható jackpot periódus")

# ============================================================================
# 3. JACKPOT RATIO BECSLÉSE
# ============================================================================

print("\n" + "-"*80)
print("JACKPOT RATIO BECSLÉS")
print("-"*80)

# A 2-3-4 találatosok arányából becsüljük a jackpot ratiót
# Már korábban kiszámoltuk:
print(f"\nKorábban becsült kifizetési arányok (szelvényenként):")
print(f"  2 találat: {payout_2:.6f}× ({payout_2*100:.4f}%)")
print(f"  3 találat: {payout_3:.6f}× ({payout_3*100:.4f}%)")
print(f"  4 találat: {payout_4:.6f}× ({payout_4*100:.4f}%)")
print(f"  Összesen: {(payout_2+payout_3+payout_4):.6f}× ({(payout_2+payout_3+payout_4)*100:.2f}%)")

# Ha a teljes kifizetési arány ~50%, akkor:
total_payout_ratio = 0.50
jackpot_ratio_estimated = total_payout_ratio - (payout_2 + payout_3 + payout_4)

print(f"\n→ Becsült jackpot ratio: {jackpot_ratio_estimated:.6f}× ({jackpot_ratio_estimated*100:.2f}%)")

# Most használjuk ezt a jackpot periódusok elemzésére
jackpot_df['estimated_players'] = (
    jackpot_df['jackpot_relative'] / (jackpot_df['weeks'] * jackpot_ratio_estimated)
)

print(f"\nJackpot alapú játékosszám becslések:")
print(jackpot_df[['year', 'weeks', 'jackpot_relative', 'estimated_players']].describe())

# ============================================================================
# 4. ÖSSZEHASONLÍTÁS A 2-3-4 TALÁLATOS BECSLÉSEKKEL
# ============================================================================

print("\n" + "="*80)
print("BECSLÉSEK ÖSSZEHASONLÍTÁSA")
print("="*80)

# Készítsünk egy közös dataframe-et időbeli elemzéshez
comparison_df = huzasok_clean.copy()

comparison_df['date_parsed'] = pd.to_datetime(
    comparison_df['date'],
    format="%Y.%m.%d.",
    errors="coerce"
)

# Jackpot alapú becslések hozzáadása
# Minden periódushoz rendeljük hozzá a becsült játékosszámot
comparison_df['players_from_jackpot'] = np.nan

for _, period in jackpot_df.iterrows():
    for idx in period['period_indices']:
        if idx < len(comparison_df):
            comparison_df.loc[idx, 'players_from_jackpot'] = period['estimated_players']

# Csak azokat nézzük, ahol mindkét becslés van
both_df = comparison_df.dropna(subset=['estimated_players', 'players_from_jackpot'])

print(f"\n{len(both_df)} sorsolás, ahol mindkét becslés elérhető")

if len(both_df) > 10:
    from scipy.stats import pearsonr, spearmanr
    
    corr_pearson, p_pearson = pearsonr(
        both_df['estimated_players'], 
        both_df['players_from_jackpot']
    )
    corr_spearman, p_spearman = spearmanr(
        both_df['estimated_players'], 
        both_df['players_from_jackpot']
    )
    
    print(f"\nKorreláció a két becslés között:")
    print(f"  Pearson: {corr_pearson:.3f} (p={p_pearson:.4f})")
    print(f"  Spearman: {corr_spearman:.3f} (p={p_spearman:.4f})")
    
    # Átlagos eltérés
    ratio = both_df['players_from_jackpot'] / both_df['estimated_players']
    print(f"\nJackpot becslés / 2-3-4 becslés arány:")
    print(f"  Medián: {ratio.median():.3f}")
    print(f"  Átlag: {ratio.mean():.3f}")
    print(f"  Szórás: {ratio.std():.3f}")

# ============================================================================
# 5. REGRESSZIÓS MODELL - IDŐBELI TREND
# ============================================================================

print("\n" + "="*80)
print("IDŐBELI TRENDEK - REGRESSZIÓS MODELL")
print("="*80)

# Dátum konverzió
comparison_df['date_parsed'] = pd.to_datetime(
    comparison_df['date'], 
    format='%Y.%m.%d.', 
    errors='coerce'
)

# Numerikus dátum (napok 1957 óta)
min_date = comparison_df['date_parsed'].min()
comparison_df['days_since_start'] = (
    comparison_df['date_parsed'] - min_date
).dt.days

# Csak a legutóbbi 20 év (jobb minőségű adatok)
recent_df = comparison_df[comparison_df['year'] >= 2005].copy()

print(f"\nModellezés 2005 óta: {len(recent_df)} sorsolás")

# Lineáris regresszió
from sklearn.linear_model import LinearRegression

X = recent_df[['days_since_start']].values
y_234 = recent_df['estimated_players'].values
y_jackpot = recent_df['players_from_jackpot'].values

# Csak ahol van adat
mask_234 = ~np.isnan(y_234)
mask_jackpot = ~np.isnan(y_jackpot)

if mask_234.sum() > 100:
    model_234 = LinearRegression()
    model_234.fit(X[mask_234], y_234[mask_234])
    
    trend_234 = model_234.coef_[0]
    intercept_234 = model_234.intercept_
    
    print(f"\n2-3-4 alapú becslés trend:")
    print(f"  Változás: {trend_234:.2f} játékos/nap")
    print(f"  Éves változás: {trend_234 * 365:,.0f} játékos/év")
    print(f"  2005 becsült: {intercept_234:,.0f} játékos")
    print(f"  2025 becsült: {intercept_234 + trend_234 * 365 * 20:,.0f} játékos")

if mask_jackpot.sum() > 20:
    model_jackpot = LinearRegression()
    model_jackpot.fit(X[mask_jackpot], y_jackpot[mask_jackpot])
    
    trend_jackpot = model_jackpot.coef_[0]
    intercept_jackpot = model_jackpot.intercept_
    
    print(f"\nJackpot alapú becslés trend:")
    print(f"  Változás: {trend_jackpot:.2f} játékos/nap")
    print(f"  Éves változás: {trend_jackpot * 365:,.0f} játékos/év")
    print(f"  2005 becsült: {intercept_jackpot:,.0f} játékos")
    print(f"  2025 becsült: {intercept_jackpot + trend_jackpot * 365 * 20:,.0f} játékos")

# ============================================================================
# 6. VIZUALIZÁCIÓ
# ============================================================================

print("\n" + "="*80)
print("VIZUALIZÁCIÓ LÉTREHOZÁSA")
print("="*80)

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# ============================================================================
# PLOT 1: Időbeli összehasonlítás
# ============================================================================

ax1 = axes[0]

# 2-3-4 alapú becslés
mask = ~comparison_df['estimated_players'].isna() & (comparison_df['year'] >= 2000)
ax1.scatter(
    comparison_df.loc[mask, 'date_parsed'],
    comparison_df.loc[mask, 'estimated_players'],
    alpha=0.3, s=10, label='2-3-4 találat alapú', color='blue'
)

# Jackpot alapú becslés
mask_jp = ~comparison_df['players_from_jackpot'].isna() & (comparison_df['year'] >= 2000)
ax1.scatter(
    comparison_df.loc[mask_jp, 'date_parsed'],
    comparison_df.loc[mask_jp, 'players_from_jackpot'],
    alpha=0.6, s=30, label='Jackpot felhalmozódás alapú', 
    color='red', marker='x'
)

# Mozgóátlag (simítás)
if mask.sum() > 50:
    smooth_234 = comparison_df.loc[mask, 'estimated_players'].rolling(
        window=52, center=True, min_periods=10
    ).mean()
    ax1.plot(
        comparison_df.loc[mask, 'date_parsed'],
        smooth_234,
        color='darkblue', linewidth=2, label='2-3-4 trend (52 hét)',
        alpha=0.7
    )

ax1.set_xlabel('Dátum', fontsize=11)
ax1.set_ylabel('Becsült játékosszám', fontsize=11)
ax1.set_title('Játékosszám becslések összehasonlítása (2000-)', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# ============================================================================
# PLOT 2: Arány az idő függvényében
# ============================================================================

ax2 = axes[1]

if len(both_df) > 0:
    both_df_sorted = both_df.sort_values('date_parsed')
    ratio = both_df_sorted['players_from_jackpot'] / both_df_sorted['estimated_players']
    
    ax2.scatter(
        both_df_sorted['date_parsed'],
        ratio,
        alpha=0.5, s=20, color='purple'
    )
    
    # Mozgóátlag
    ratio_smooth = ratio.rolling(window=20, center=True, min_periods=5).mean()
    ax2.plot(
        both_df_sorted['date_parsed'],
        ratio_smooth,
        color='darkviolet', linewidth=2, label='Trend (20 hét)',
        alpha=0.8
    )
    
    ax2.axhline(y=1.0, color='red', linestyle='--', linewidth=1.5, 
                label='Egyezés (1.0)', alpha=0.7)
    
    ax2.set_xlabel('Dátum', fontsize=11)
    ax2.set_ylabel('Jackpot becslés / 2-3-4 becslés', fontsize=11)
    ax2.set_title('Becslési módszerek aránya', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# ============================================================================
# PLOT 3: Évenkénti átlagok
# ============================================================================

ax3 = axes[2]

# Évenkénti aggregálás
yearly_234 = comparison_df[comparison_df['year'] >= 2000].groupby('year').agg({
    'estimated_players': 'mean'
}).reset_index()

yearly_jackpot = comparison_df[comparison_df['year'] >= 2000].groupby('year').agg({
    'players_from_jackpot': 'mean'
}).reset_index()

ax3.plot(
    yearly_234['year'], 
    yearly_234['estimated_players'],
    marker='o', linewidth=2, markersize=6,
    label='2-3-4 alapú (éves átlag)', color='blue'
)

ax3.plot(
    yearly_jackpot['year'], 
    yearly_jackpot['players_from_jackpot'],
    marker='x', linewidth=2, markersize=8,
    label='Jackpot alapú (éves átlag)', color='red'
)

ax3.set_xlabel('Év', fontsize=11)
ax3.set_ylabel('Átlagos játékosszám', fontsize=11)
ax3.set_title('Éves átlagos játékosszám becslések', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('player_estimates_comparison.png', dpi=150, bbox_inches='tight')
print("\n✓ Grafikon mentve: player_estimates_comparison.png")
plt.show()

# ============================================================================
# 7. VÉGSŐ KONKLÚZIÓK
# ============================================================================

print("\n" + "="*80)
print("KONKLÚZIÓK")
print("="*80)

recent_players_234 = recent_df['estimated_players'].mean()
recent_players_jackpot = recent_df['players_from_jackpot'].mean()

print(f"""
ÁTLAGOS JÁTÉKOSSZÁM (2005 óta):
  • 2-3-4 találat alapú:    {recent_players_234:>12,.0f} játékos
  • Jackpot felhalmozódás:  {recent_players_jackpot:>12,.0f} játékos
  • Eltérés:                {abs(recent_players_234 - recent_players_jackpot):>12,.0f} ({abs(recent_players_234 - recent_players_jackpot)/recent_players_234*100:.1f}%)

MEGBÍZHATÓSÁG:
  • 2-3-4 alapú: ✓✓✓ Nagy mintaszám, stabil
  • Jackpot alapú: ✓✓ Kis mintaszám, nagyobb variancia
  
AJÁNLÁS:
  A 2-3-4 találatos becslés megbízhatóbb a nagyobb mintaszám miatt.
  A jackpot alapú becslés validációként használható.
  
JACKPOT RATIO:
  Becsült érték: {jackpot_ratio_estimated*100:.2f}% a heti bevételből
  Ez alapján: {estimated_total_jackpot:.1f}× átlagos jackpot
              ≈ {estimated_total_jackpot * 400:,.0f} Ft
""")

In [ ]:
print(huzasok_clean)
huzasok_clean["EV_rel"] = huzasok_clean["rel_fours_prize"] * P_4 + huzasok_clean["rel_threes_prize"] * P_3 + huzasok_clean["rel_twos_prize"] * P_2
huzasok_clean['EV_rel'].describe()

In [ ]:
# Leghatékonyabb számok (EV a héten) regresszió

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# Tegyük fel, hogy a húzások adatai már a 'huzasok' DataFrame-ben vannak
# és van egy 'numbers' oszlop, ami a kihúzott számokat tartalmazza listaként
# Ha nincs ilyen, akkor azt először létre kell hozni

# Példa adatszerkezet kialakítása (ha nincs numbers oszlop)
# Ez feltételezi, hogy a számok külön oszlopokban vannak: number1, number2, ...
numbers_columns = ['n1', 'n2', 'n3', 'n4', 'n5']

# Ha a számok más formátumban vannak, először át kell alakítani
# Példa: ha a számok szöveges formában vannak egy oszlopban
# huzasok_clean['numbers'] = huzasok_clean['numbers_str'].apply(lambda x: list(map(int, x.split(','))))

# Dummy változók létrehozása minden számhoz (1-90)
def create_number_dummies(df, numbers_col='numbers'):
    """Létrehoz dummy változókat minden lottószámhoz"""
    dummies_data = []
    
    for idx, row in df.iterrows():
        # Feltételezzük, hogy numbers lista formájában van
        numbers = row[numbers_col]
        dummy_row = {f'num_{i}': 1 if i in numbers else 0 for i in range(1, 91)}
        dummies_data.append(dummy_row)
    
    return pd.DataFrame(dummies_data)

# Ha még nincs numbers oszlop, hozzuk létre a meglévő számokból
# Ez feltételezi, hogy a számok külön oszlopokban vannak
if 'numbers' not in huzasok_clean.columns and all(col in huzasok_clean.columns for col in numbers_columns):
    huzasok_clean['numbers'] = huzasok_clean[numbers_columns].values.tolist()

# Dummy változók létrehozása
print("Dummy változók létrehozása...")
X_dummies = create_number_dummies(huzasok_clean, 'numbers')
y = huzasok_clean['EV_rel'].values

# Lineáris regresszió a dummy változókkal
print("Regresszió futtatása...")
model = LinearRegression()
model.fit(X_dummies, y)

# Együtthatók összegyűjtése
coef_df = pd.DataFrame({
    'number': range(1, 91),
    'coefficient': model.coef_[:90]  # Csak az első 90 együttható (1-90 számok)
})

# Intercept (konstans tag) - ez az alap várható érték
print(f"\nAlap várható érték (intercept): {model.intercept_:.6f}")

# Együtthatók rendezése
coef_df = coef_df.sort_values('coefficient', ascending=False)

# Top 20 legnépszerűbb/leghatékonyabb szám (pozitív együttható = jobb EV)
print("\n" + "="*60)
print("TOP 20 LEGNÉPSZERŰBB/HATÉKONYABB SZÁM:")
print("="*60)
for i, (_, row) in enumerate(coef_df.head(20).iterrows(), 1):
    print(f"{i:2d}. {row['number']} - együttható: {row['coefficient']:.6f}")

# Top 20 legkevésbé népszerű/hatékony szám (negatív együttható = rosszabb EV)
print("\n" + "="*60)
print("TOP 20 LEGKEVÉSBÉ NÉPSZERŰ/HATÉKONY SZÁM:")
print("="*60)
for i, (_, row) in enumerate(coef_df.tail(20).iterrows(), 1):
    print(f"{i:2d}. {row['number']} - együttható: {row['coefficient']:.6f}")

# További statisztikák
print("\n" + "="*60)
print("EGYÜTTHATÓK STATISZTIKÁI:")
print("="*60)
print(f"Átlagos együttható: {coef_df['coefficient'].mean():.6f}")
print(f"Szórás: {coef_df['coefficient'].std():.6f}")
print(f"Maximum: {coef_df['coefficient'].max():.6f} (szám: {coef_df.loc[coef_df['coefficient'].idxmax(), 'number']})")
print(f"Minimum: {coef_df['coefficient'].min():.6f} (szám: {coef_df.loc[coef_df['coefficient'].idxmin(), 'number']})")

# Modell teljesítményének értékelése
y_pred = model.predict(X_dummies)
r_squared = model.score(X_dummies, y)
print(f"\nModell R² értéke: {r_squared:.6f}")
print(f"MAE: {np.mean(np.abs(y - y_pred)):.6f}")
print(f"MSE: {np.mean((y - y_pred)**2):.6f}")

# Számok gyakorisága
print("\n" + "="*60)
print("SZÁMOK GYAKORISÁGA (TOP 10 LEGGYAKORIBB):")
print("="*60)
# Számoljuk meg, hogy az egyes számok hányszor szerepeltek
if 'numbers' in huzasok_clean.columns:
    all_numbers = [num for sublist in huzasok_clean['numbers'] for num in sublist]
    freq_series = pd.Series(all_numbers).value_counts().sort_values(ascending=False)
    
    for i, (num, freq) in enumerate(freq_series.head(10).items(), 1):
        percentage = (freq / len(huzasok_clean)) * 100
        print(f"{i:2d}. {num:2d} - {freq:4d} alkalommal ({percentage:.1f}% a húzásokból)")

In [ ]:
# Regresszió pred
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Adatok előkészítése
# Feltételezzük, hogy már van huzasok_clean DataFrame-ed

# Ellenőrizzük, hogy van-e numbers oszlop
if 'numbers' not in huzasok_clean.columns:
    # Ha nincs, hozzuk létre a számokból
    # Módosítsd a oszlopneveket a saját adataidnak megfelelően
    numbers_columns = ['number1', 'number2', 'number3', 'number4', 'number5']
    if all(col in huzasok_clean.columns for col in numbers_columns):
        huzasok_clean['numbers'] = huzasok_clean[numbers_columns].values.tolist()
    else:
        raise ValueError("Nincs 'numbers' oszlop, és nem találhatóak a számok külön oszlopokban sem.")

# 2. Dummy változók létrehozása
def create_number_dummies_from_list(numbers_list):
    """Létrehoz dummy változókat egy szám listából"""
    return {f'num_{i}': 1 if i in numbers_list else 0 for i in range(1, 91)}

# Teljes dummy DataFrame létrehozása
print("Dummy változók létrehozása...")
dummy_data = []
for numbers in huzasok_clean['numbers']:
    dummy_data.append(create_number_dummies_from_list(numbers))

X = pd.DataFrame(dummy_data)
y = huzasok_clean['EV_rel'].values

# 3. Modell felépítése és tanítása
print("Modell tanítása...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

# 4. Modell kiértékelése
print("\n" + "="*60)
print("MODELL KIÉRTÉKELÉSE")
print("="*60)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

print(f"Tanító halmaz:")
print(f"  R²: {r2_score(y_train, y_pred_train):.6f}")
print(f"  MAE: {mean_absolute_error(y_train, y_pred_train):.6f}")
print(f"  MSE: {mean_squared_error(y_train, y_pred_train):.6f}")

print(f"\nTeszt halmaz:")
print(f"  R²: {r2_score(y_test, y_pred_test):.6f}")
print(f"  MAE: {mean_absolute_error(y_test, y_pred_test):.6f}")
print(f"  MSE: {mean_squared_error(y_test, y_pred_test):.6f}")

print(f"\nAlap várható érték (intercept): {model.intercept_:.6f}")

# 5. EV becslő funkció
def estimate_ev_for_numbers(numbers_list, model=model):
    """
    Becsüli a várható értéket adott számokhoz
    
    Parameters:
    -----------
    numbers_list : list
        5 elemű lista a kihúzott számokkal (1-90)
    model : LinearRegression
        A betanított modell
    
    Returns:
    --------
    float : becsült EV_rel érték
    """
    # Dummy változók létrehozása
    dummy_input = create_number_dummies_from_list(numbers_list)
    
    # DataFrame-é alakítás
    input_df = pd.DataFrame([dummy_input])
    
    # Sorrend biztosítása (ugyanaz, mint a tanításnál)
    input_df = input_df[X.columns]
    
    # Becslés
    predicted_ev = model.predict(input_df)[0]
    
    return predicted_ev

# 6. Példa becslések
print("\n" + "="*60)
print("PÉLDA BECSLÉSEK")
print("="*60)

# Néhány példa számkombináció
example_combinations = [
    [1, 2, 3, 4, 5],  # Nagyon alacsony számok
    [86, 87, 88, 89, 90],  # Nagyon magas számok
    [7, 14, 21, 28, 35],  # 7-es lépésköz
    [10, 20, 30, 40, 50],  # 10-es lépésköz
    [5, 15, 25, 35, 45],  # 10-es lépésköz páratlanok
]

for i, combo in enumerate(example_combinations, 1):
    ev_estimate = estimate_ev_for_numbers(combo)
    print(f"{i}. {sorted(combo)} -> EV becslés: {ev_estimate:.6f}")

# 7. Valós húzásokkal való összehasonlítás
print("\n" + "="*60)
print("VALÓS HÚZÁSOK BECSLÉSE ÉS ÖSSZEHASONLÍTÁS")
print("="*60)

# Vegyünk 5 véletlen húzást a teszt halmazból
test_indices = np.random.choice(len(X_test), min(5, len(X_test)), replace=False)

for i, idx in enumerate(test_indices, 1):
    # Valós számok és EV
    real_idx = X_test.index[idx]
    real_numbers = huzasok_clean.loc[real_idx, 'numbers']
    real_ev = huzasok_clean.loc[real_idx, 'EV_rel']
    
    # Becsült EV
    predicted_ev = y_pred_test[idx]
    
    # Hiba
    error = predicted_ev - real_ev
    
    print(f"{i}. Valós számok: {sorted(real_numbers)}")
    print(f"   Valós EV: {real_ev:.6f}, Becsült EV: {predicted_ev:.6f}")
    print(f"   Hiba: {error:.6f} ({abs(error/real_ev*100):.2f}%)")
    print()

# 8. Legjobb és legrosszabb EV-jű kombinációk keresése
print("\n" + "="*60)
print("LEGJOBB ÉS LEGROSSZABB SZÁMKOMBINÁCIÓK KERESÉSE")
print("="*60)

# Együtthatók rendezése
coef_df = pd.DataFrame({
    'number': range(1, 91),
    'coefficient': model.coef_[:90]
}).sort_values('coefficient', ascending=False)

# Top 5 legjobb és legrosszabb szám
best_numbers = coef_df.head(5)['number'].tolist()
worst_numbers = coef_df.tail(5)['number'].tolist()

print(f"Top 5 legjobb EV-t növelő szám: {best_numbers}")
print(f"Top 5 legrosszabb EV-t csökkentő szám: {worst_numbers}")

# Legjobb kombináció (csak a legjobb számokból)
best_combo = best_numbers[:5]
best_ev = estimate_ev_for_numbers(best_combo)
print(f"\nLegjobb kombináció (csak top 5 számból): {sorted(best_combo)}")
print(f"Becsült EV: {best_ev:.6f}")

# Legrosszabb kombináció (csak a legrosszabb számokból)
worst_combo = worst_numbers[:5]
worst_ev = estimate_ev_for_numbers(worst_combo)
print(f"\nLegrosszabb kombináció (csak bottom 5 számból): {sorted(worst_combo)}")
print(f"Becsült EV: {worst_ev:.6f}")

# Véletlen kombináció
random_combo = np.random.choice(range(1, 91), 5, replace=False).tolist()
random_ev = estimate_ev_for_numbers(random_combo)
print(f"\nVéletlen kombináció: {sorted(random_combo)}")
print(f"Becsült EV: {random_ev:.6f}")

# 9. EV becslés alapján kombinációk generálása
print("\n" + "="*60)
print("OPTIMÁLIS KOMBINÁCIÓK GENERÁLÁSA")
print("="*60)

def generate_combinations_by_ev(num_combinations=10):
    """
    Generál EV alapján rangsorolt kombinációkat
    """
    from itertools import combinations
    import random
    
    # Csak a legjobb 20 számból generáljunk kombinációkat a gyorsaság kedvéért
    top_numbers = coef_df.head(20)['number'].tolist()
    
    combinations_list = []
    
    # Véletlen kombinációk generálása
    for _ in range(num_combinations):
        combo = random.sample(top_numbers, 5)
        ev = estimate_ev_for_numbers(combo)
        combinations_list.append((sorted(combo), ev))
    
    # Rendezés EV szerint csökkenő sorrendben
    combinations_list.sort(key=lambda x: x[1], reverse=True)
    
    return combinations_list

# Generáljunk néhány optimális kombinációt
optimal_combinations = generate_combinations_by_ev(5)

print("Top 5 optimális kombináció (becsült EV alapján):")
for i, (combo, ev) in enumerate(optimal_combinations, 1):
    print(f"{i}. {combo} -> EV: {ev:.6f}")

# 10. Mentés a későbbi használatra
import joblib
import os

# Modell mentése
model_data = {
    'model': model,
    'feature_columns': X.columns.tolist(),
    'intercept': model.intercept_,
    'coefficients': dict(zip(range(1, 91), model.coef_[:90]))
}

# Mappa létrehozása, ha nem létezik
os.makedirs('lotto_model', exist_ok=True)

# Mentés
joblib.dump(model_data, 'lotto_model/ev_predictor.pkl')
print("\nModell mentve: 'lotto_model/ev_predictor.pkl'")

# Együtthatók CSV-be mentése
coef_df.to_csv('lotto_model/number_coefficients.csv', index=False)
print("Együtthatók mentve: 'lotto_model/number_coefficients.csv'")

In [ ]:
# P2*rel_prize2 + P3*rel_prize3 + P4*rel_prize4 + P5*rel_prize5 > 1 
# rel_prize5 (= prize5 / ticket_price) > (1 - szum(Pi*rel_prizei))/P5

def calculate_required_jackpot(rel_ev, ticket_price=400):
    from scipy.special import comb

    # Konstansok
    TOTAL_NUMBERS = 90
    NUMBERS_DRAWN = 5

    # Kombinációs valószínűségek
    TOTAL_COMBINATIONS = comb(TOTAL_NUMBERS, NUMBERS_DRAWN, exact=True)
    P_5 = 1 / TOTAL_COMBINATIONS

    jackpot = (1 - rel_ev) / P_5 * ticket_price / 1_000_000
    print(f"Required jackpot: {jackpot:.0f} million HUF")

    return jackpot

calculate_required_jackpot(0.603025)

# 1. 60, 87, 80, 88, 40
# 2. 61, 46, 82, 89, 90
# 3. 85, 65, 86, 62, 30